In [32]:
import numpy
import pandas as pd
import pathlib

from windpowerlib import WindTurbine, ModelChain

curr_dir = pathlib.Path(".")
data_dir = curr_dir / "wind-data" / "WTK-LED"


In [88]:
buses = [d.stem for d in list(data_dir.glob("raw/bus*"))]
for bus in buses:
    bus_data_files = sorted(list((data_dir / "raw" / bus).glob("*")))
    for f in bus_data_files:
        # Read in / re-format data
        df = pd.read_csv(f, index_col=[0])[["wind speed at 100m (m/s)"]]
        df.index = pd.to_datetime(df.index)
        df.columns = pd.MultiIndex.from_tuples([('wind_speed', 100)])
        # Get normalized power output
        turbine = WindTurbine(
            hub_height=100,
            turbine_type='V112/3000'
        )
        mc = ModelChain(turbine)
        mc.run_model(df)
        df_power = mc.power_output / turbine.nominal_power
        # Save normalized profile
        df_power.name = "profile"
        (data_dir / "profiles" / bus).mkdir(exist_ok=True, parents=True)
        df_power.to_csv(data_dir / "profiles" / bus / f.stem)